# Limpieza de datos

In [1]:
import pandas as pd  # librería para trabajar con tablas

from config import CRUDO, PROCESADO, ANIOS  # traigo de config.py las carpetas datos/crudo y datos/procesado, y los años

In [2]:
# Creo una función que lee un año y verifica que se haya leído completo

def leer_anio(anio):  # recibe un año
    """Lee la base anual de importaciones de un año y la verifica contra el archivo de control.

    Supone que el zip ya se descomprimió en datos/crudo/imports_{anio}/
    y que los archivos se llaman impoaAA.csv e itotaAA.csv (AA = últimos dos dígitos del año).
    """
    carpeta = CRUDO / f"imports_{anio}"  # carpeta donde está descomprimido ese año
    aa = str(anio)[-2:]  # paso el año a texto y me quedo con los dos últimos caracteres

    # Leo el archivo de datos
    datos = pd.read_csv(  # pd.read_csv lee un archivo CSV y lo convierte en una tabla 
        carpeta / f"impoa{aa}.csv",  # ruta del archivo
        sep=";",  # las columnas están separadas por punto y coma
        decimal=",",  # los decimales usan coma, no punto
        encoding="latin1",  # la codificación del archivo; sin esto, la ñ de "Año" sale rota
        dtype={"NCM": str, "Porg": str},  # leo los códigos como texto para no perder los ceros a la izquierda
    )
    # Leo el archivo de control, que trae los totales que publica el INDEC
    control = pd.read_csv(
        carpeta / f"itota{aa}.csv",  # por ejemplo itota21.csv
        sep=";",
        decimal=",",
        encoding="latin1",
    )

    #hallazgo 
    # Entre 2019 y 2021 el INDEC usa "(u$s)" y desde 2022 "(USD)": unifico al formato nuevo
    datos.columns = datos.columns.str.replace("(u$s)", "(USD)", regex=False)  # cambio el texto en los nombres de las columnas
    control.columns = control.columns.str.replace("(u$s)", "(USD)", regex=False)  # regex=False: busco el texto tal cual

    # Verificación contra los totales de control del INDEC
    assert len(datos) == control.loc[0, "Total de registros"], f"{anio}: no coincide la cantidad de filas"  # misma cantidad de filas
    assert abs(datos["CIF(USD)"].sum() - control.loc[0, "Total CIF(USD)"]) < 1, f"{anio}: no coincide el total CIF"  # mismo total, con margen de 1 dólar

    return datos  # devuelvo la tabla del año

Reviso los nombres de las columnas de cada año. Entre 2019 y 2021 el INDEC usa (u$s) y desde 2022 (USD), así que en la función de lectura los unifico.

In [3]:
#miro los nombres de las columnas de cada año, sin leer los datos

for anio in ANIOS:  # repito para cada año
    aa = str(anio)[-2:]  # los dos últimos dígitos del año: 2021 -> "21"
    carpeta = CRUDO / f"imports_{anio}"  # carpeta de ese año
    datos_cols = pd.read_csv(carpeta / f"impoa{aa}.csv", sep=";", encoding="latin1", nrows=0).columns  # nrows=0: leo solo los títulos
    control_cols = pd.read_csv(carpeta / f"itota{aa}.csv", sep=";", encoding="latin1", nrows=0).columns  # lo mismo con el archivo de control
    print(anio)  # muestro el año
    print("  datos:  ", list(datos_cols))  # y los nombres de las columnas de cada archivo
    print("  control:", list(control_cols))

2019
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2020
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2021
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2022
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(USD)', 'Flete(USD)', 'Seguro(USD)', 'CIF(USD)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de regis

In [4]:
bases = []  # lista vacía donde voy a guardar la tabla de cada año
for anio in ANIOS:  # repito para cada año
    bases.append(leer_anio(anio))  # leo el año con mi función (que además lo verifica) y lo agrego a la lista
    print(anio, "ok")  # si llegó acá, pasaron los assert de ese año

impo = pd.concat(bases, ignore_index=True)  # apilo las 7 tablas una debajo de la otra en una sola tabla
print(impo.shape)  # muestro (filas, columnas)
impo.head()  # muestro las primeras 5 filas

2019 ok
2020 ok
2021 ok
2022 ok
2023 ok
2024 ok
2025 ok
(490822, 8)


,Año,NCM,Porg,Pnet(kg),FOB(USD),Flete(USD),Seguro(USD),CIF(USD)
0,2019,01012100,200,9800.0,354774.00,45675.01,3636.86,404085.87
1,2019,01012100,203,11350.0,174203.39,19332.02,1683.94,195219.35
2,2019,01012100,208,11000.0,136291.00,6294.63,2725.82,145311.45
3,2019,01012100,212,8300.0,449989.50,46382.80,3226.60,499598.90
4,2019,01012100,218,500.0,35000.00,1724.64,367.25,37091.89


In [5]:
print(impo.dtypes)  # muestro el tipo de dato de cada columna

assert set(impo["Año"]) == set(ANIOS), "Faltan o sobran años"  #están los años que pedí
assert not impo.duplicated(["Año", "NCM", "Porg"]).any(), "Hay filas repetidas"  #no hay dos filas con el mismo año, producto y país
assert impo["CIF(USD)"].ge(0).all(), "Hay montos negativos"  #todos los montos son mayores o iguales a cero

Año              int64
NCM                str
Porg               str
Pnet(kg)       float64
FOB(USD)       float64
Flete(USD)     float64
Seguro(USD)    float64
CIF(USD)       float64
dtype: object


In [6]:
#Leo el nomenclador del INDEC: dice qué país es cada código

paises = pd.read_csv(
    CRUDO / "paises.csv",  # el archivo que bajé de la página del INDEC
    sep=";",  # columnas separadas por punto y coma
    dtype=str,  # leo todas las columnas como texto (son códigos y nombres)
    keep_default_na=False,  # para que el código "NA" de Namibia no se lea como dato faltante
)
paises = paises.rename(columns={"_id": "Porg"})  # le cambio el nombre a la columna para que coincida con la base de importaciones

print(paises.shape)  # (280, 3): 280 códigos y 3 columnas
paises.head()  # muestro las primeras 5 filas

(280, 3)


,Porg,iso2,nombre
0,301,AF,Afganistán
1,401,AL,Albania
2,404,AD,Andorra
3,149,AO,Angola
4,237,AG,Antigua y Barbuda


In [7]:
assert paises["Porg"].is_unique, "Hay códigos de país repetidos en el nomenclador"
assert (paises["iso2"] != "").all(), "Hay países sin código ISO"
assert set(impo["Porg"]).issubset(set(paises["Porg"])), "Hay códigos en la base que no están en el nomenclador"

## Expresión regular 1: entradas que no son países

In [19]:
import re  # librería de expresiones regulares

# Función que dice si un código ISO es una entrada especial (no un país)
def es_codigo_especial(iso2):  # recibe un código de dos letras, por ejemplo "BR"
    """Indica si un código ISO de dos letras NO corresponde a un país.

    La norma ISO 3166 reserva AA, QM a QZ, XA a XZ y ZZ para usos propios.
    El INDEC los usa para entradas como 'Confidencial', 'Indeterminado',
    'Resto' o 'Sin declarar'. Supone que el código viene en mayúsculas.
    """
    # ^ y $: el código entero tiene que ser una de las opciones; | significa "o"
    return re.search(r"^(AA|Q[M-Z]|X[A-Z]|ZZ)$", iso2) is not None  # True si es especial, False si es un país

In [20]:
casos = {
    "ZZ": True,   # Confidencial
    "XR": True,   # Indeterminado (continente)
    "QZ": True,   # Indeterminado (América)
    "BR": False,  # Brasil
    "CN": False,  # China
    "QA": False,  # Qatar: empieza con Q pero no está en el rango reservado
    "NA": False,  # Namibia
    "": False,
}
for texto, esperado in casos.items():
    obtenido = es_codigo_especial(texto)
    assert obtenido == esperado, f"{texto}: se esperaba {esperado} y se obtuvo {obtenido}"
print("Los", len(casos), "casos pasan")

Los 8 casos pasan


Además de ‘Confidencial’ e ‘Indeterminado’, aparecen códigos que no son países, como ‘Calamares’, ‘Mar Territorial Argentino’ o ‘Productos Argentinos a Embajadas Extranjeras’. Todos usan códigos ISO reservados.

In [ ]:
paises["especial"] = paises["iso2"].map(es_codigo_especial)  # aplico la función a cada código y guardo el resultado en una columna nueva
print(paises["especial"].value_counts())  # cuento cuántos True (especiales) y cuántos False (países) hay
paises.loc[paises["especial"], ["Porg", "iso2", "nombre"]]  # muestro solo las filas especiales, con esas tres columnas

especial
False    247
True      33
Name: count, dtype: int64


,Porg,iso2,nombre
33,994,XO,Calamares
46,999,ZZ,Confidencial
74,996,XP,Exportaciones e Importaciones Simplificadas
98,398,XC,Indeterminado ( Asia )
99,298,QZ,Indeterminado (América)
100,498,XF,Indeterminado (Europa)
101,598,XM,Indeterminado (Oceanía)
102,998,XR,Indeterminado (continente)
103,198,QQ,Indeterminado (África)
145,295,QW,Mar Territorial Argentino y/o Zona Económica E...


In [ ]:
filas_antes = len(impo)  # guardo cuántas filas tengo antes de unir, para comparar después

# Uno cada fila de importaciones con su país del nomenclador, usando el código Porg
impo = impo.merge(paises, on="Porg", how="left", validate="m:1", indicator=True)

print("Filas antes:", filas_antes, "| Filas después:", len(impo))  # comparo la cantidad de filas
print(impo["_merge"].value_counts())  # cuento cuántas filas encontraron pareja y cuántas no

assert len(impo) == filas_antes, "El merge cambió la cantidad de filas"  # no se agregaron ni se perdieron filas
assert (impo["_merge"] == "both").all(), "Hay códigos de país sin pareja en el nomenclador"  # todas encontraron su país

impo = impo.drop(columns="_merge")  # borro la columna _merge, que ya cumplió su función

Filas antes: 490822 | Filas después: 490822
_merge
both          490822
left_only          0
right_only         0
Name: count, dtype: int64


In [ ]:
print(impo["especial"].value_counts())  # cuántas filas son especiales (True) y cuántas son países (False)

cif_total = impo.groupby("Año")["CIF(USD)"].sum()  # total importado de cada año
cif_especial = impo[impo["especial"]].groupby("Año")["CIF(USD)"].sum()  # total importado de entradas especiales, por año

print((cif_especial / cif_total * 100).round(2))  # porcentaje que representan las especiales en cada año, con 2 decimales

especial
False    490663
True        159
Name: count, dtype: int64
Año
2019    1.19
2020    1.07
2021    1.05
2022    1.09
2023    1.07
2024    0.94
2025    1.59
Name: CIF(USD), dtype: float64


Las entradas que no son países (“Confidencial”, “Indeterminado”, etc.) son 159 filas y representan entre 0,9% y 1,6% del valor importado cada año. No se pueden ubicar en el mapa, así que las dejo afuera del análisis por país, pero informo su peso. Al ser menos del 2% en todos los años, no deberían cambiar las conclusiones.

## Expresión regular 2: formato de los códigos de producto

In [ ]:
# Función que dice si un código de producto nomenclador comun del mercosur (NCM) tiene el formato correcto
def es_ncm_valido(ncm):  # recibe un código de producto, por ejemplo "01012100"
    """Indica si un código NCM tiene el formato correcto: exactamente 8 dígitos.

    Sirve para detectar códigos que perdieron el cero inicial (por haberse leído
    como número) o que tienen letras o espacios. Supone que el código es texto.
    """
    # ^ empieza, \d{8} son 8 dígitos, $ termina. el texto tiene que ser exactamente 8 dígitos
    return re.search(r"^\d{8}$", ncm) is not None  # True si es válido, False si no

In [ ]:
#casos de prueba: el código y lo que la función tendría que devolver
casos_ncm = {
    "01012100": True,    # caballos reproductores, con su cero inicial
    "87032310": True,    # autos
    "1012100": False,    # el mismo código si perdió el cero: 7 dígitos
    "0101210A": False,   # tiene una letra
    "01012100 ": False,  # tiene un espacio al final
    "": False,           # texto vacío
}

for texto, esperado in casos_ncm.items():  # recorro cada par código-resultado esperado
    obtenido = es_ncm_valido(texto)  # aplico la función
    assert obtenido == esperado, f"{texto}: se esperaba {esperado} y se obtuvo {obtenido}"  # compruebo que dé lo esperado
print("Los", len(casos_ncm), "casos pasan")  # si llegó acá, pasaron todos

Los 6 casos pasan


In [ ]:
validos = impo["NCM"].map(es_ncm_valido)  # aplico la función a cada código, una columna de True y False
print("Códigos válidos:", validos.sum(), "| No válidos:", (~validos).sum())  # cuento cuántos cumplen y cuántos no
assert validos.all(), "Hay códigos NCM con formato incorrecto"  # compruebo que todos sean válidos

Códigos válidos: 490822 | No válidos: 0


Los 490.822 códigos tienen 8 dígitos, por lo que no hubo que corregir ninguno.

In [22]:
#saco las entradas que no son países (Confidencial, Indeterminado, etc.)
impo_paises = impo[~impo["especial"]]  # me quedo con las filas donde especial es False (el ~ invierte)

#sumo el valor importado por año y país
por_pais = impo_paises.groupby(["Año", "iso2"], as_index=False)["CIF(USD)"].sum()  # agrupo por año y país y sumo el CIF

print(por_pais.shape)  
por_pais.head()  # muestro las primeras filas

(1272, 3)


,Año,iso2,CIF(USD)
0,2019,AD,2.820360e+03
1,2019,AE,1.860961e+08
2,2019,AF,2.665620e+04
3,2019,AG,7.798900e+02
4,2019,AL,7.226068e+05


In [23]:
total_anio = por_pais.groupby("Año")["CIF(USD)"].sum()  # total importado de todos los países, para cada año
por_pais["participacion"] = por_pais["CIF(USD)"] / por_pais["Año"].map(total_anio) * 100  # % de cada país sobre el total de su año

por_pais.sort_values(["Año", "participacion"], ascending=[True, False]).head(10)  # ordeno por año y, dentro de cada año, de mayor a menor participación

,Año,iso2,CIF(USD),participacion
22,2019,BR,1.015933e+10,20.928787
36,2019,CN,9.258744e+09,19.073529
176,2019,US,6.217043e+09,12.807455
44,2019,DE,2.765873e+09,5.697852
143,2019,PY,1.646757e+09,3.392412
21,2019,BO,1.368970e+09,2.820155
165,2019,TH,1.184467e+09,2.440068
83,2019,IT,1.125974e+09,2.319569
122,2019,MX,1.125233e+09,2.318042
53,2019,ES,9.772216e+08,2.013131


In [24]:
assert not por_pais.duplicated(["Año", "iso2"]).any(), "Hay un país repetido en el mismo año"  # cada país aparece una sola vez por año
assert set(por_pais["Año"]) == set(ANIOS), "Faltan años"  # están todos los años

suma = por_pais.groupby("Año")["participacion"].sum()  # sumo las participaciones de cada año
assert ((suma - 100).abs() < 0.001).all(), "Las participaciones no suman 100% en algún año"  # en cada año tienen que sumar 100%

PROCESADO.mkdir(parents=True, exist_ok=True)  # creo la carpeta datos/procesado si no existe
por_pais.to_csv(PROCESADO / "importaciones_por_pais.csv", index=False)  # guardo la tabla final como CSV